In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# ==============================================================================
# Legal Document Summarization using T5 and Hugging Face Transformers
# ==============================================================================
#
# This script provides an end-to-end implementation for fine-tuning a T5
# transformer model for the task of legal text summarization.
#
# It's designed to be run in a Google Colab environment and leverages
# the Hugging Face libraries (Transformers, Datasets, Evaluate).
#
# The process includes:
# 1. Installing and importing necessary libraries.
# 2. Loading and preparing the 'in_abs' dataset.
# 3. Tokenizing the documents and summaries for the T5 model.
# 4. Setting up the Trainer with training arguments and evaluation metrics.
# 5. Running the fine-tuning process.
# 6. Saving the trained model.
# 7. Demonstrating how to use the fine-tuned model for inference.
#
# ==============================================================================


# ==============================================================================
# 1. SETUP AND INSTALLATIONS
# ==============================================================================
# First, we need to install the required libraries from Hugging Face, along with
# a library for calculating the ROUGE metric, which is standard for summarization.
# The 'sacrebleu' library is also often used for translation/summarization tasks.
# The 'ipywidgets' is needed to render progress bars in Colab.
print("Step 1: Installing required libraries...")
# Using -q for a quieter installation
!pip install -q transformers[torch] datasets evaluate rouge_score sacrebleu ipywidgets

print("Libraries installed successfully.")

# Import all the necessary modules
import torch
import numpy as np
import datasets
import evaluate
import os
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# Disable Weights & Biases logging
# The Hugging Face Trainer automatically tries to log to wandb if it's installed.
# This line prevents that and stops it from asking for an API key.
os.environ["WANDB_DISABLED"] = "true"

# ==============================================================================
# 2. CONFIGURATION AND MODEL SELECTION
# ==============================================================================
# We define our model checkpoint. 't5-small' is a good starting point as it
# trains relatively quickly and requires less memory. For better performance,
# you could switch to 't5-base' or 't5-large', but they will require more
# computational resources (and time).
MODEL_CHECKPOINT = "t5-small"
# T5 models require a prefix for the task they are performing. For summarization,
# "summarize: " is a standard choice.
PREFIX = "summarize: "

# Let's also set a device to use GPU if available (which it should be in Colab).
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


# ==============================================================================
# 3. DATASET LOADING AND PREPARATION
# ==============================================================================
print("\nStep 3: Loading and preparing the dataset...")

# Load the dataset from the Hugging Face Hub.
# CORRECTION: The correct path is "percins/IN-ABS".
raw_datasets = load_dataset("percins/IN-ABS")

print("Dataset loaded. Here is an example from the training set:")
print(raw_datasets["train"][0])


# ==============================================================================
# 4. TOKENIZATION
# ==============================================================================
print("\nStep 4: Setting up the tokenizer and preprocessing the data...")

# Load the tokenizer associated with our chosen model checkpoint.
# The tokenizer is responsible for converting raw text into a format the
# model can understand (i.e., token IDs).
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# We need a function to preprocess our data. This function will:
# 1. Add the "summarize: " prefix to the input documents.
# 2. Tokenize both the input documents and the target summaries.
# 3. Set the tokenized summaries as the 'labels' for the model to learn from.
def preprocess_function(examples):
    # Prepare the inputs by adding the prefix
    # CORRECTION: The key for the main document is 'text', not 'original_text'.
    inputs = [PREFIX + doc for doc in examples["text"]]

    # Tokenize the inputs. We truncate them to a max length to handle very long
    # documents. 1024 is a reasonable length for 't5-small'.
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)

    # Tokenize the target summaries (labels).
    # The 'with tokenizer.as_target_tokenizer():' block ensures that the
    # tokenizer handles the target sequence appropriately.
    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True)

    # The model expects the target IDs to be in the 'labels' field.
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply the preprocessing function to all splits of the dataset.
# The `batched=True` argument processes multiple examples at once for efficiency.
tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

print("Data preprocessing and tokenization complete.")


# ==============================================================================
# 5. MODEL LOADING AND TRAINING SETUP
# ==============================================================================
print("\nStep 5: Loading the model and setting up for training...")

# Load the pretrained T5 model for sequence-to-sequence tasks.
# We move the model to the GPU if one is available.
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT).to(DEVICE)

# The DataCollatorForSeq2Seq is a utility that will dynamically pad the
# input and label tensors in each batch to the maximum length in that batch.
# This is more efficient than padding all examples to a global maximum length.
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# Load the ROUGE metric for evaluation.
rouge = evaluate.load("rouge")

# Define a function to compute metrics during evaluation.
# This will be called at the end of each evaluation phase.
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Decode the generated predictions and the true labels back to text.
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # The -100 in labels is a convention to ignore padding tokens.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute the ROUGE scores.
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    # Extract key results and also calculate the length of predictions.
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    # Return the metrics, rounded for readability.
    return {k: round(v, 4) for k, v in result.items()}


# ==============================================================================
# 6. FINE-TUNING THE MODEL
# ==============================================================================
print("\nStep 6: Fine-tuning the model...")

# Define the training arguments. These control various aspects of the training
# process, such as the number of epochs, batch size, learning rate, etc.
training_args = Seq2SeqTrainingArguments(
    output_dir="legal_t5_summarizer",
    # CORRECTION: The argument name was changed in newer versions of the library.
    eval_strategy="epoch",  # Evaluate at the end of each epoch
    learning_rate=2e-5,
    per_device_train_batch_size=4, # Reduced batch size for Colab
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=3,          # Only keep the best 3 models
    num_train_epochs=3,          # A small number of epochs for demonstration
    predict_with_generate=True,  # Necessary for summarization evaluation
    fp16=True,                   # Use mixed-precision training for speedup on GPU
    push_to_hub=False,           # Set to True if you want to upload to Hugging Face Hub
)

# Initialize the Trainer. The Trainer class handles the entire training and
# evaluation loop for us.
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Start the training process!
trainer.train()

print("Model fine-tuning complete.")


# ==============================================================================
# 7. SAVE THE MODEL
# ==============================================================================
print("\nStep 7: Saving the fine-tuned model...")

# Define a path to save the final model and tokenizer
model_save_path = "/content/drive/MyDrive/fine_tuned_legal_summarizer"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"Model saved to {model_save_path}")


# ==============================================================================
# 8. GENERATE SUMMARIES (INFERENCE)
# ==============================================================================
print("\nStep 8: Generating summaries with the new model...")

# Let's see how our model performs on a new, unseen document.
# We'll take an example from the test set.
test_document = raw_datasets["test"][5]["text"]
reference_summary = raw_datasets["test"][5]["summary"]

print("-" * 50)
print(f"Original Document:\n{test_document[:1000]}...") # Print first 1000 chars
print("-" * 50)
print(f"Reference Summary:\n{reference_summary}")
print("-" * 50)


# Create a summarization pipeline with our fine-tuned model.
# This is the easiest way to perform inference.
# Load the model and tokenizer from the saved path
saved_tokenizer = AutoTokenizer.from_pretrained(model_save_path)
saved_model = AutoModelForSeq2SeqLM.from_pretrained(model_save_path).to(DEVICE)

def summarize_text(document):
    """
    Function to generate a summary for a given text document using the
    fine-tuned model.
    """
    # Add the prefix and tokenize the document
    inputs = saved_tokenizer(
        PREFIX + document,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(DEVICE)

    # Generate the summary
    summary_ids = saved_model.generate(
        inputs["input_ids"],
        num_beams=4,      # Beam search can improve results
        max_length=150,   # Set a max length for the summary
        early_stopping=True
    )

    # Decode the generated IDs to get the text
    generated_summary = saved_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )
    return generated_summary

# Generate the summary for our test document
model_generated_summary = summarize_text(test_document)

print(f"Model Generated Summary:\n{model_generated_summary}")
print("-" * 50)
print("\nDemonstration complete.")

